<a href="https://colab.research.google.com/github/D-F-V/Project_brief/blob/main/exploracion_madrid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploración inicial del mercado residencial de Madrid

**Dataset:** idealista18, objeto Madrid_Sale.
**Periodo:** cuatro trimestres de 2018.

Este notebook recoge la exploración inicial de los datos utilizada para preparar el primer Project Brief. Su objetivo es conocer las características de los anuncios, detectar problemas de calidad y justificar las decisiones posteriores del proyecto.

Los precios corresponden a anuncios de venta, no a operaciones de compraventa cerradas. Además, los datos son de 2018, por lo que no describen el mercado actual.

Durante esta exploración se conservan los registros originales. Se comprueba la información, se retira la geometría redundante de la copia tabular y se marcan algunos casos para su revisión. Detectar un dato sospechoso no implica eliminarlo automáticamente.

La limpieza definitiva, el entrenamiento y la evaluación de los modelos se realizan en el notebook de la segunda entrega.

**Cómo ejecutar el notebook:** ejecutar las celdas de arriba abajo, porque utilizan variables creadas en pasos anteriores. La descarga inicial requiere conexión a Internet.

## 1. Entorno y procedencia de los datos

### 1.1. Instalación de la herramienta de lectura

El dataset original está guardado en formato RDA, utilizado por R. Instalamos la librería `rdata` para poder leerlo desde Python.

En el siguiente paso se mostrarán las versiones de las librerías utilizadas para dejar constancia del entorno de ejecución.


In [ ]:
!pip install -q rdata

### 1.2. Librerías y versiones

Importamos las librerías necesarias para cargar y analizar los datos. También mostramos las versiones de Python y de las principales herramientas utilizadas.

Registrar estas versiones ayuda a reproducir el análisis y a investigar posibles diferencias si el notebook se ejecuta en otro entorno.

In [ ]:
import hashlib
import platform
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import rdata

print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("rdata:", rdata.__version__)

Python: 3.13.18
pandas: 2.2.3
numpy: 2.4.6
rdata: 1.1.0


### 1.3. Descarga del dataset

Descargamos el archivo `Madrid_Sale.rda` del repositorio público `paezha/idealista18`.

El enlace utiliza un commit concreto, es decir, una versión identificada del repositorio. Esto permite mantener la misma fuente aunque el repositorio cambie posteriormente.

El código crea la carpeta `data` si no existe y guarda el archivo dentro. Si ya está descargado, reutiliza el archivo local para evitar descargarlo de nuevo.

En el siguiente apartado registraremos su procedencia y su huella digital para identificar el archivo utilizado.


In [ ]:
REPO = "paezha/idealista18"
COMMIT = "e9ef80618f8a9cc28f672ac138f298bbd519918e"
RUTA_EN_REPO = "data/Madrid_Sale.rda"
URL = f"https://raw.githubusercontent.com/{REPO}/{COMMIT}/{RUTA_EN_REPO}"

CARPETA_DATOS = Path("data")
CARPETA_DATOS.mkdir(exist_ok=True)
RUTA_LOCAL = CARPETA_DATOS / "Madrid_Sale.rda"

if RUTA_LOCAL.exists():
    print("El archivo ya existe, no se vuelve a descargar:", RUTA_LOCAL)
else:
    urllib.request.urlretrieve(URL, RUTA_LOCAL)
    print("Descargado en:", RUTA_LOCAL)

Descargado en: data/Madrid_Sale.rda


### 1.4. Registro de procedencia

Registramos la dirección de descarga, el commit utilizado, el tamaño del archivo y su huella digital SHA-256.

La huella permite identificar el contenido del archivo y comprobar si coincide con otra copia. En este código se calcula y se muestra, pero no se compara automáticamente con un valor esperado.

La fecha registrada corresponde a la última modificación del archivo local. No representa la fecha de publicación del dataset y puede cambiar si el archivo se copia o se descarga de nuevo.

También anotamos la licencia ODbL 1.0 indicada en el README del repositorio de origen. Esta información permite documentar de dónde proceden los datos y bajo qué licencia se distribuyen.

In [ ]:
contenido = RUTA_LOCAL.read_bytes()
fecha_descarga = datetime.fromtimestamp(RUTA_LOCAL.stat().st_mtime, tz=timezone.utc)

procedencia = {
    "url": URL,
    "repositorio": f"https://github.com/{REPO}",
    "commit": COMMIT,
    "ruta_en_repositorio": RUTA_EN_REPO,
    "licencia_declarada": "ODbL 1.0",
    "fecha_descarga_utc": fecha_descarga.strftime("%Y-%m-%d %H:%M:%S"),
    "tamano_bytes": len(contenido),
    "sha256": hashlib.sha256(contenido).hexdigest(),
}

for clave, valor in procedencia.items():
    print(f"{clave}: {valor}")

url: https://raw.githubusercontent.com/paezha/idealista18/e9ef80618f8a9cc28f672ac138f298bbd519918e/data/Madrid_Sale.rda
repositorio: https://github.com/paezha/idealista18
commit: e9ef80618f8a9cc28f672ac138f298bbd519918e
ruta_en_repositorio: data/Madrid_Sale.rda
licencia_declarada: ODbL 1.0
fecha_descarga_utc: 2026-11-16 14:53:22
tamano_bytes: 6738247
sha256: c50e281ee699a8a523a5505dd64eb1d5aa0e3ba712092355850ed6d9f0d38a20


## 2. Estructura y cobertura de los datos

En este apartado comprobamos qué contiene el archivo, cuántas filas y columnas tiene y qué periodos recoge. También revisamos los tipos de datos y la información geográfica.

Estas comprobaciones permiten conocer la estructura del dataset antes de analizar su calidad.

### 2.1. Lectura del archivo original

Cargamos el archivo RDA y revisamos los objetos que contiene para localizar `Madrid_Sale`, que recoge los anuncios de venta utilizados en este proyecto.

Durante la lectura pueden aparecer advertencias relacionadas con la conversión de objetos de R a Python. Estas advertencias no demuestran por sí solas que la carga haya fallado: en los siguientes pasos comprobaremos las dimensiones, las columnas y algunos registros para verificar el resultado.

In [ ]:
objeto_rda = rdata.read_rda(RUTA_LOCAL)

print(type(objeto_rda))
print(list(objeto_rda.keys()))

/usr/local/lib/python3.13/dist-packages/rdata/conversion/_conversion.py:918: UserWarning: Missing constructor for R class "bbox". The underlying R object is returned instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/rdata/conversion/_conversion.py:918: UserWarning: Missing constructor for R class "crs". The underlying R object is returned instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/rdata/conversion/_conversion.py:918: UserWarning: Missing constructor for R class "XY". The constructor for class "POINT" will be used instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/rdata/conversion/_conversion.py:918: UserWarning: Missing constructor for R class "POINT". The constructor for class "sfg" will be used instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/rdata/conversion/_conversion.py:918: UserWarning: Missing constructor for R class "sfg". The underlying R object is returned instead.
  warnings.warn(
/usr/local/lib/python3.

<class 'dict'>
['Madrid_Sale']


### 2.2. Dimensiones del dataset

Comprobamos el número de filas y columnas del objeto `Madrid_Sale` y consultamos los nombres de sus variables.

El archivo cargado contiene 94.815 registros y 42 columnas, incluida la geometría. Cada fila corresponde a un registro de anuncio; no necesariamente a un inmueble distinto, porque un mismo identificador puede aparecer varias veces.

Más adelante comprobaremos si la geometría aporta información adicional a las columnas de latitud y longitud. Si resulta redundante, la copia tabular quedará con 41 columnas.

In [ ]:
madrid = objeto_rda["Madrid_Sale"]

print(type(madrid))
print("Filas y columnas:", madrid.shape)
print("Primeros valores del índice:", list(madrid.index[:5]))

<class 'pandas.core.frame.DataFrame'>
Filas y columnas: (94815, 42)
Primeros valores del índice: [1, 2, 3, 4, 5]


### 2.3. Tipos de datos

Revisamos cómo se representa cada variable en Python: números, categorías o texto.

Esta comprobación ayuda a detectar columnas que podrían necesitar una conversión antes del análisis. Que una variable contenga números no significa que represente una cantidad: algunos valores son códigos de categorías, como la ubicación del piso o la calidad catastral.

En este paso solo inspeccionamos los tipos de datos. Las decisiones sobre cómo tratar cada variable se tomarán después, teniendo en cuenta su significado.

In [ ]:
resumen_tipos = madrid.dtypes.astype(str).rename("tipo").to_frame()
resumen_tipos.index.name = "columna"
print(resumen_tipos.to_string())
print()
print(madrid.dtypes.astype(str).value_counts().to_string())

                                   tipo
columna                                
ASSETID                        category
PERIOD                            Int32
PRICE                           float64
UNITPRICE                       float64
CONSTRUCTEDAREA                   Int32
ROOMNUMBER                        Int32
BATHNUMBER                        Int32
HASTERRACE                        Int32
HASLIFT                           Int32
HASAIRCONDITIONING                Int32
AMENITYID                         Int32
HASPARKINGSPACE                   Int32
ISPARKINGSPACEINCLUDEDINPRICE     Int32
PARKINGSPACEPRICE                 Int32
HASNORTHORIENTATION               Int32
HASSOUTHORIENTATION               Int32
HASEASTORIENTATION                Int32
HASWESTORIENTATION                Int32
HASBOXROOM                        Int32
HASWARDROBE                       Int32
HASSWIMMINGPOOL                   Int32
HASDOORMAN                        Int32
HASGARDEN                         Int32


### 2.4. Vista de los primeros registros

Mostramos algunos registros para comprobar cómo se ha cargado la información y observar el contenido de las columnas.

Esta vista permite reconocer variables como el identificador del anuncio, el periodo, el precio, la superficie y las características del inmueble.

La muestra sirve como comprobación inicial, pero no permite sacar conclusiones sobre todo el dataset. Los valores ausentes, las repeticiones y los posibles errores se revisarán después utilizando el conjunto completo.

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

madrid.head(3).T

,1,2,3
ASSETID,A15019136831406238029,A6677225905472065344,A13341979748618524775
PERIOD,201803,201803,201803
PRICE,126000.0,235000.0,373000.0
UNITPRICE,2680.851064,4351.851852,4973.333333
CONSTRUCTEDAREA,47,54,75
ROOMNUMBER,1,1,2
BATHNUMBER,1,1,1
HASTERRACE,0,0,0
HASLIFT,1,0,0
HASAIRCONDITIONING,1,0,1


### 2.5. Inspección inicial de la geometría

El dataset incluye una columna de geometría y dos columnas con las coordenadas: `LONGITUDE` y `LATITUDE`.

Inspeccionamos la geometría de un registro y la comparamos con sus coordenadas para comprobar cómo está representada la ubicación.

Esta primera comparación permite detectar una posible duplicación de información, pero no demuestra que ocurra en todos los registros. En el siguiente paso realizaremos la comprobación sobre el conjunto completo.

In [ ]:
print(type(madrid["geometry"].iloc[0]))
print(madrid["geometry"].iloc[0])

<class 'numpy.ndarray'>
[-3.76693283 40.36248477]


### 2.6. Comprobación de la geometría en todo el dataset

Comparamos las coordenadas de la geometría con las columnas `LONGITUDE` y `LATITUDE` en todos los registros.

La comprobación muestra que ambas representaciones coinciden. Por tanto, la geometría resulta redundante para este análisis y se retira de la copia tabular, conservando las columnas de longitud y latitud.

La tabla queda con 94.815 registros y 41 columnas. Este cambio no elimina anuncios ni corrige sus coordenadas: únicamente evita conservar dos representaciones de la misma información.

In [ ]:
coords_geom = np.vstack(madrid["geometry"].to_numpy())

dif_lon = np.abs(coords_geom[:, 0] - madrid["LONGITUDE"].to_numpy(dtype=float))
dif_lat = np.abs(coords_geom[:, 1] - madrid["LATITUDE"].to_numpy(dtype=float))

print("Forma de las coordenadas de geometry:", coords_geom.shape)
print("Máxima diferencia en longitud:", dif_lon.max())
print("Máxima diferencia en latitud:", dif_lat.max())
print("Filas con diferencia mayor que 1e-6:", int(((dif_lon > 1e-6) | (dif_lat > 1e-6)).sum()))

Forma de las coordenadas de geometry: (94815, 2)
Máxima diferencia en longitud: 0.0
Máxima diferencia en latitud: 0.0
Filas con diferencia mayor que 1e-6: 0


### 2.7. Cobertura por trimestre

Contamos los registros de cada periodo para conocer cómo se distribuyen los anuncios a lo largo de 2018.

Los códigos `201803`, `201806`, `201809` y `201812` identifican los cuatro cortes trimestrales disponibles. No representan una serie mensual completa.

La cantidad de registros varía entre trimestres. Esta diferencia debe tenerse en cuenta al comparar resultados: un periodo con más anuncios no demuestra por sí solo un aumento de las ventas, ya que los datos recogen anuncios y no operaciones cerradas.

In [ ]:
datos = madrid.drop(columns="geometry")

por_periodo = datos.groupby("PERIOD").agg(
    filas=("ASSETID", "size"),
    assetid_distintos=("ASSETID", "nunique"),
)
print(por_periodo.to_string())
print()
print("Periodos distintos:", sorted(datos["PERIOD"].dropna().unique().tolist()))
print("Periodos ausentes:", int(datos["PERIOD"].isna().sum()))

        filas  assetid_distintos
PERIOD                          
201803  21920              17622
201806  12652              12518
201809  15973              14246
201812  44270              31418

Periodos distintos: [201803, 201806, 201809, 201812]
Periodos ausentes: 0


### Conclusión sobre la estructura y la cobertura

El archivo original contiene 94.815 registros y 42 columnas. Tras retirar la geometría redundante de la copia tabular, se conservan los mismos registros y 41 columnas.

Los datos abarcan cuatro cortes trimestrales de 2018. La cobertura es desigual: el último trimestre concentra 44.270 registros, por lo que tiene un mayor peso en los análisis realizados sobre el conjunto completo.

A continuación revisaremos los valores ausentes y los identificadores repetidos para conocer mejor la calidad de los datos.

## 3. Valores ausentes e identificadores repetidos

Revisamos qué información falta y si existen registros o identificadores repetidos. Estas comprobaciones ayudan a decidir cómo preparar los datos para el análisis y el modelado.

### 3.1. Valores ausentes por variable

Calculamos la cantidad y el porcentaje de valores ausentes en cada columna.

El porcentaje permite distinguir entre variables con ausencias puntuales y otras con información incompleta en una parte importante del dataset. Por ejemplo, `CONSTRUCTIONYEAR` presenta un 58,93 % de valores ausentes, por lo que su uso requerirá especial atención.

En esta fase no rellenamos valores ni eliminamos registros. Primero documentamos las ausencias; su tratamiento definitivo se justificará durante la preparación de los datos para el modelo.

In [ ]:
nulos = datos.isna().sum()
resumen_nulos = pd.DataFrame({
    "nulos": nulos,
    "porcentaje": (nulos / len(datos) * 100).round(2),
})
print(resumen_nulos[resumen_nulos["nulos"] > 0].sort_values("nulos", ascending=False).to_string())
print()
print("Columnas con algún nulo:", int((nulos > 0).sum()), "de", datos.shape[1])

                    nulos  porcentaje
CONSTRUCTIONYEAR    55873       58.93
FLATLOCATIONID       6387        6.74
FLOORCLEAN           3846        4.06
CADASTRALQUALITYID      1        0.00

Columnas con algún nulo: 4 de 41


### 3.2. Duplicados exactos e identificadores repetidos

Comprobamos dos situaciones diferentes:

- **Duplicados exactos:** filas que coinciden en todas sus columnas.
- **Identificadores repetidos:** registros que comparten el mismo `ASSETID`, aunque pueden presentar diferencias en otras variables.

No se detectan duplicados exactos. Sin embargo, algunos identificadores aparecen varias veces, por lo que el número de registros es superior al de inmuebles distintos.

Compartir identificador no basta para eliminar una fila. En los siguientes pasos revisaremos en qué periodos aparecen estas repeticiones y qué información cambia entre sus registros.

Si el recuento muestra categorías con frecuencia cero, corresponden a categorías conservadas en la variable que no aparecen en las filas actuales; no representan inmuebles adicionales.

In [ ]:
print("Filas totales:", len(datos))
print("Duplicados exactos (fila completa):", int(datos.duplicated().sum()))
print("ASSETID distintos:", datos["ASSETID"].nunique())
print("Filas cuyo ASSETID aparece más de una vez:", int(datos["ASSETID"].duplicated(keep=False).sum()))
print("Filas en pares ASSETID-PERIOD repetidos:", int(datos.duplicated(subset=["ASSETID", "PERIOD"], keep=False).sum()))
print()
veces = datos["ASSETID"].value_counts()
print("Cuántos ASSETID aparecen 1, 2, 3... veces:")
print(veces.value_counts().sort_index().to_string())

Filas totales: 94815
Duplicados exactos (fila completa): 0
ASSETID distintos: 75804
Filas cuyo ASSETID aparece más de una vez: 32840
Filas en pares ASSETID-PERIOD repetidos: 32840

Cuántos ASSETID aparecen 1, 2, 3... veces:
count
0      1058
1     61975
2      9842
3      3098
4       673
5       150
6        51
7        11
8         1
9         2
11        1


### 3.3. Repeticiones por periodo

Comprobamos si los identificadores repetidos aparecen dentro del mismo trimestre o en trimestres diferentes.

En este dataset, ningún `ASSETID` aparece en más de un trimestre. Por tanto, las repeticiones detectadas se producen dentro de un mismo periodo.

Este resultado impide utilizar el identificador para seguir la evolución de un mismo inmueble entre trimestres. Las comparaciones temporales describen conjuntos de anuncios diferentes, no necesariamente cambios de precio de las mismas viviendas.

Más adelante, al dividir los datos para entrenar y evaluar el modelo, volveremos a comprobar que ningún identificador se comparte entre conjuntos.

In [ ]:
periodos_por_id = datos.groupby("ASSETID", observed=True)["PERIOD"].nunique()
print("Periodos distintos por ASSETID (cuántos ASSETID hay de cada caso):")
print(periodos_por_id.value_counts().sort_index().to_string())
print()

filas_por_par = datos.groupby(["ASSETID", "PERIOD"], observed=True).size()
print("Máximo de filas por par ASSETID-PERIOD:", int(filas_por_par.max()))

Periodos distintos por ASSETID (cuántos ASSETID hay de cada caso):
PERIOD
1    75804

Máximo de filas por par ASSETID-PERIOD: 11


### 3.4. Diferencias entre registros del mismo identificador

Revisamos qué variables presentan valores diferentes entre los registros que comparten un mismo `ASSETID`.

Esta comprobación permite distinguir una repetición del identificador de una duplicación completa del anuncio. Aunque dos registros correspondan al mismo identificador, pueden contener información diferente.

Las diferencias no permiten determinar por sí solas cuál de los registros es correcto ni en qué orden se produjeron los cambios. Por eso, no seleccionamos automáticamente una fila ni eliminamos las demás.

En el siguiente paso observaremos algunos ejemplos para entender mejor estas diferencias.

In [ ]:
repetidos = datos[datos["ASSETID"].duplicated(keep=False)].copy()
repetidos["ASSETID"] = repetidos["ASSETID"].astype(str)

columnas = [c for c in repetidos.columns if c != "ASSETID"]
difiere = (
    repetidos.groupby("ASSETID")[columnas]
    .nunique(dropna=False)
    .gt(1)
    .sum()
    .sort_values(ascending=False)
)

print("Grupos de ASSETID repetidos:", repetidos["ASSETID"].nunique())
print()
print("En cuántos grupos difiere cada columna:")
print(difiere.to_string())

Grupos de ASSETID repetidos: 13829

En cuántos grupos difiere cada columna:
DISTANCE_TO_CITY_CENTER          13825
DISTANCE_TO_CASTELLANA           13825
LONGITUDE                        13825
DISTANCE_TO_METRO                13825
LATITUDE                         13825
UNITPRICE                        13548
PRICE                            13539
CONSTRUCTEDAREA                    974
CONSTRUCTIONYEAR                   466
AMENITYID                          364
ISPARKINGSPACEINCLUDEDINPRICE      363
HASPARKINGSPACE                    363
FLOORCLEAN                         282
HASTERRACE                         276
ROOMNUMBER                         253
FLATLOCATIONID                     237
HASSOUTHORIENTATION                236
HASDOORMAN                         231
HASEASTORIENTATION                 226
HASWARDROBE                        205
HASAIRCONDITIONING                 202
HASWESTORIENTATION                 200
PARKINGSPACEPRICE                  188
BUILTTYPEID_3              

### 3.5. Ejemplos de identificadores repetidos

Mostramos algunos registros que comparten identificador para observar directamente sus similitudes y diferencias.

Estos ejemplos ayudan a interpretar la comprobación anterior, pero no permiten generalizar a todos los identificadores repetidos ni determinar qué registro debería conservarse.

En esta exploración mantenemos los registros. Las repeticiones se tendrán en cuenta al preparar las divisiones del modelo y al evaluar sus resultados, porque los inmuebles con más registros pueden tener mayor peso.

In [ ]:
ejemplo_id = repetidos["ASSETID"].iloc[0]
print("ASSETID del ejemplo:", ejemplo_id)
repetidos[repetidos["ASSETID"] == ejemplo_id].T

ASSETID del ejemplo: A4775182175615276542


,4,3344,78833
ASSETID,A4775182175615276542,A4775182175615276542,A4775182175615276542
PERIOD,201803,201803,201803
PRICE,284000.0,258000.0,257000.0
UNITPRICE,5916.666667,5733.333333,6425.0
CONSTRUCTEDAREA,48,45,40
ROOMNUMBER,1,1,1
BATHNUMBER,1,1,1
HASTERRACE,0,0,0
HASLIFT,1,1,1
HASAIRCONDITIONING,1,1,1


### Conclusión sobre ausencias y repeticiones

Las principales ausencias se concentran en `CONSTRUCTIONYEAR` (58,93 %), `FLATLOCATIONID` (6,74 %) y `FLOORCLEAN` (4,06 %). También falta un valor en `CADASTRALQUALITYID`.

Los 94.815 registros contienen 75.804 identificadores distintos. No hay duplicados exactos, pero 13.829 identificadores aparecen varias veces y reúnen 32.840 registros.

Las repeticiones se producen dentro del mismo trimestre. Por tanto, no podemos seguir la evolución de una vivienda entre periodos mediante `ASSETID`.

Estas comprobaciones justifican documentar el tratamiento de las ausencias y controlar el peso de los identificadores repetidos durante el modelado.

## 4. Rangos, códigos y coherencia

Revisamos los valores de las variables para detectar posibles errores, códigos que necesiten interpretación y datos que requieran una comprobación más detallada.

Un valor poco frecuente o extremo no es necesariamente incorrecto. Antes de decidir su tratamiento, debemos considerar el significado de la variable y la documentación del dataset.

### 4.1. Resumen de las variables numéricas

Consultamos los mínimos, máximos y otros estadísticos de las columnas numéricas para conocer sus rangos.

Prestamos especial atención al precio, la superficie, los dormitorios, los baños, los años de construcción y las distancias. Esta revisión permite localizar valores sospechosos que analizaremos en los siguientes pasos.

Algunas columnas numéricas representan categorías mediante códigos. En esos casos, su media no tiene una interpretación directa y debemos revisar los valores posibles y su significado.

In [ ]:
numericas = [
    "PRICE", "UNITPRICE", "CONSTRUCTEDAREA", "ROOMNUMBER", "BATHNUMBER",
    "CONSTRUCTIONYEAR", "CADCONSTRUCTIONYEAR", "CADMAXBUILDINGFLOOR", "CADDWELLINGCOUNT",
    "DISTANCE_TO_CITY_CENTER", "DISTANCE_TO_METRO", "DISTANCE_TO_CASTELLANA",
    "PARKINGSPACEPRICE", "LONGITUDE", "LATITUDE",
]

resumen = datos[numericas].astype("float64").describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).T
print(resumen.round(2).to_string())

                           count       mean        std       min        1%        5%        25%        50%        75%         95%         99%         max
PRICE                    94815.0  396110.11  417074.41  21000.00  71000.00  97000.00  160000.00  262000.00  467000.00  1135000.00  2207000.00  8133000.00
UNITPRICE                94815.0    3661.05    1700.50    805.31   1088.89   1426.67    2240.00    3480.00    4744.62     6771.43     8584.21     9997.56
CONSTRUCTEDAREA          94815.0     101.40      67.08     21.00     30.00     40.00      62.00      83.00     117.00      225.00      367.00      985.00
ROOMNUMBER               94815.0       2.58       1.24      0.00      0.00      1.00       2.00       3.00       3.00        4.00        6.00       93.00
BATHNUMBER               94815.0       1.59       0.84      0.00      1.00      1.00       1.00       1.00       2.00        3.00        5.00       20.00
CONSTRUCTIONYEAR         38942.0    1964.69      55.89      1.00   1880.00  

### 4.2. Variables binarias

Revisamos las variables que indican la presencia o ausencia de una característica del inmueble, como ascensor, terraza o aire acondicionado.

Estas variables se representan mediante los valores `0` y `1`. Comprobamos qué valores aparecen y cuántos registros corresponden a cada uno para detectar posibles códigos inesperados.

La interpretación de cada columna debe apoyarse en la documentación del dataset. Que una variable solo contenga ceros y unos confirma su formato binario, pero no garantiza que la información de cada anuncio sea correcta.

In [ ]:
binarias = [c for c in datos.columns if c.startswith(("HAS", "IS", "BUILTTYPEID_"))]

filas = []
for c in binarias:
    s = datos[c].astype("float64")
    filas.append({
        "columna": c,
        "valor_0": int((s == 0).sum()),
        "valor_1": int((s == 1).sum()),
        "otros": int((s.notna() & ~s.isin([0, 1])).sum()),
        "nulos": int(s.isna().sum()),
    })

print(pd.DataFrame(filas).set_index("columna").to_string())

                               valor_0  valor_1  otros  nulos
columna                                                      
HASTERRACE                       61131    33684      0      0
HASLIFT                          28862    65953      0      0
HASAIRCONDITIONING               52281    42534      0      0
HASPARKINGSPACE                  73472    21343      0      0
ISPARKINGSPACEINCLUDEDINPRICE    73472    21343      0      0
HASNORTHORIENTATION              84592    10223      0      0
HASSOUTHORIENTATION              72427    22388      0      0
HASEASTORIENTATION               75602    19213      0      0
HASWESTORIENTATION               80739    14076      0      0
HASBOXROOM                       70272    24543      0      0
HASWARDROBE                      40580    54235      0      0
HASSWIMMINGPOOL                  80767    14048      0      0
HASDOORMAN                       71151    23664      0      0
HASGARDEN                        77319    17496      0      0
ISDUPLEX

### 4.3. Códigos de las variables categóricas

Revisamos los códigos y sus frecuencias para conocer las categorías presentes en el dataset.

Aunque estas variables contienen números, sus valores pueden representar características del inmueble y no cantidades. Por ejemplo, `FLATLOCATIONID` distingue entre ubicación exterior e interior.

Interpretamos los códigos utilizando la documentación del dataset. Cuando su significado no está suficientemente descrito, conservamos el código y señalamos esa limitación, sin asignarle una interpretación inventada.

También revisamos los valores de `FLOORCLEAN`. Los códigos especiales requieren atención: no debemos asumir que todos representan plantas ordinarias sin comprobar su significado.

In [ ]:
codigos = ["AMENITYID", "FLOORCLEAN", "FLATLOCATIONID", "CADASTRALQUALITYID"]

for c in codigos:
    print(c)
    print(datos[c].value_counts(dropna=False).sort_index().to_string())
    print()

AMENITYID
AMENITYID
1     1317
2     4699
3    88799

FLOORCLEAN
FLOORCLEAN
-1        936
0       10112
1       20336
2       17044
3       14754
4       11635
5        6162
6        3733
7        2447
8        1495
9         814
10        450
11       1051
<NA>     3846

FLATLOCATIONID
FLATLOCATIONID
1       76324
2       12104
<NA>     6387

CADASTRALQUALITYID
CADASTRALQUALITYID
0         377
1         629
2        2703
3       12634
4       24673
5       20725
6       20491
7       10428
8        1527
9         627
<NA>        1



### 4.4. Coherencia geográfica

Revisamos los rangos de longitud, latitud y distancia para detectar ubicaciones que puedan resultar incoherentes con el ámbito de Madrid.

Las distancias al centro y a la Castellana se expresan en kilómetros, según la documentación del dataset. Es importante comprobar las unidades antes de interpretar sus valores.

Los registros con coordenadas o distancias inusuales se consideran candidatos a revisión. Un valor extremo no demuestra por sí solo que exista un error ni justifica eliminar automáticamente el anuncio.

Además, la fuente indica que las coordenadas están perturbadas, por lo que no deben interpretarse como direcciones exactas de los inmuebles.

In [ ]:
col = "DISTANCE_TO_CITY_CENTER"

for umbral in [12, 15, 20, 30, 50, 100]:
    print(f"Filas con {col} > {umbral} km:", int((datos[col] > umbral).sum()))

print()
lejanas = datos[datos[col] > 20].sort_values(col, ascending=False)
ver = ["ASSETID", "PERIOD", "LONGITUDE", "LATITUDE", col, "DISTANCE_TO_METRO", "UNITPRICE", "CONSTRUCTEDAREA"]
print(lejanas[ver].head(15).to_string())

Filas con DISTANCE_TO_CITY_CENTER > 12 km: 723
Filas con DISTANCE_TO_CITY_CENTER > 15 km: 1
Filas con DISTANCE_TO_CITY_CENTER > 20 km: 1
Filas con DISTANCE_TO_CITY_CENTER > 30 km: 1
Filas con DISTANCE_TO_CITY_CENTER > 50 km: 1
Filas con DISTANCE_TO_CITY_CENTER > 100 km: 1

                    ASSETID  PERIOD  LONGITUDE   LATITUDE  DISTANCE_TO_CITY_CENTER  DISTANCE_TO_METRO    UNITPRICE  CONSTRUCTEDAREA
84001  A7501251371917084834  201806  -2.753303  36.756391               415.752584         399.477366  1317.241379              145


### 4.5. Dormitorios y baños

Revisamos los valores y las frecuencias de `ROOMNUMBER` y `BATHNUMBER` para detectar cantidades inusuales o posibles problemas de información.

Según la documentación, `ROOMNUMBER` representa el número de dormitorios. Un valor de cero puede corresponder a un estudio, por lo que no se considera automáticamente un error.

Los anuncios con cero baños o un número elevado de dormitorios requieren una revisión adicional. Para interpretarlos, debemos considerar también la superficie y las demás características del inmueble.

En este paso no modificamos los valores. Más adelante comprobaremos la relación entre superficie y dormitorios para localizar posibles incoherencias.

In [ ]:
hab = datos["ROOMNUMBER"]

print("ROOMNUMBER, valores de 0 a 10:")
print(hab[hab <= 10].value_counts().sort_index().to_string())
print("Filas con 11 o más habitaciones:", int((hab >= 11).sum()))
print()
print(datos[hab >= 11][["ROOMNUMBER", "BATHNUMBER", "CONSTRUCTEDAREA", "PRICE", "UNITPRICE"]]
      .sort_values("ROOMNUMBER", ascending=False).head(10).to_string())
print()
print("Filas con ROOMNUMBER = 0, según ISSTUDIO:")
print(datos.loc[hab == 0, "ISSTUDIO"].value_counts().to_string())
print()
print("BATHNUMBER:")
print(datos["BATHNUMBER"].value_counts().sort_index().to_string())

ROOMNUMBER, valores de 0 a 10:
ROOMNUMBER
0      2745
1     13338
2     28422
3     33961
4     11674
5      3350
6       789
7       279
8       142
9        35
10       24
Filas con 11 o más habitaciones: 56

       ROOMNUMBER  BATHNUMBER  CONSTRUCTEDAREA      PRICE    UNITPRICE
17639          93           2              119   450000.0  3781.512605
84060          33           2               90   109000.0  1211.111111
68892          20          20              600   850000.0  1416.666667
545            20          16              847  2020000.0  2384.887839
25849          18           4              765  6970000.0  9111.111111
60434          18          18              455  1469000.0  3228.571429
56804          16          14              680  3375000.0  4963.235294
63967          16          10              390  1956000.0  5015.384615
11072          15          11              489  1823000.0  3728.016360
19018          15          11              489  1890000.0  3865.030675

Filas c

### 4.6. Variables de aparcamiento

Comparamos las dos variables relacionadas con el aparcamiento para comprobar si aportan información diferente.

En los registros analizados, ambas columnas presentan los mismos valores. Por tanto, incluir las dos en el modelo supondría repetir la misma información.

Esta coincidencia no aclara por sí sola su significado comercial. No debemos afirmar que la plaza está incluida en el precio o que tiene un coste adicional si la documentación no lo especifica suficientemente.

Durante la preparación del modelo se utilizará una sola de estas variables para evitar la redundancia.

In [ ]:
print(pd.crosstab(datos["HASPARKINGSPACE"], datos["ISPARKINGSPACEINCLUDEDINPRICE"],
                  rownames=["HASPARKINGSPACE"], colnames=["INCLUIDO_EN_PRECIO"]).to_string())
print()
print("¿HASPARKINGSPACE e ISPARKINGSPACEINCLUDEDINPRICE son idénticas fila a fila?:",
      bool((datos["HASPARKINGSPACE"] == datos["ISPARKINGSPACEINCLUDEDINPRICE"]).all()))
print()

es_uno = datos["PARKINGSPACEPRICE"] == 1
print(pd.crosstab(datos["HASPARKINGSPACE"], es_uno,
                  rownames=["HASPARKINGSPACE"], colnames=["PARKINGSPACEPRICE == 1"]).to_string())
print()
print("PARKINGSPACEPRICE cuando es distinto de 1:")
print(datos.loc[~es_uno, "PARKINGSPACEPRICE"].describe().round(0).to_string())

INCLUIDO_EN_PRECIO      0      1
HASPARKINGSPACE                 
0                   73472      0
1                       0  21343

¿HASPARKINGSPACE e ISPARKINGSPACEINCLUDEDINPRICE son idénticas fila a fila?: True

PARKINGSPACEPRICE == 1  False  True 
HASPARKINGSPACE                     
0                        2183  71289
1                           8  21335

PARKINGSPACEPRICE cuando es distinto de 1:
count      2191.0
mean      31110.0
std       38710.0
min           2.0
25%       16001.0
50%       25001.0
75%       40001.0
max      925001.0


### 4.7. Años de construcción

Revisamos los valores y la disponibilidad de las dos variables relacionadas con el año de construcción: `CONSTRUCTIONYEAR` y `CADCONSTRUCTIONYEAR`.

`CONSTRUCTIONYEAR` presenta numerosos valores ausentes, mientras que `CADCONSTRUCTIONYEAR` no tiene ausencias en este dataset. Esta diferencia hace que la variable catastral sea una alternativa para el modelado, aunque disponer de información completa no garantiza que todos sus valores sean correctos.

Los años anteriores a 1800 pueden corresponder a edificios históricos. Del mismo modo, los años posteriores a 2018 podrían estar relacionados con promociones futuras o con errores de registro. No los clasificamos automáticamente como incorrectos sin información adicional.

En esta fase documentamos los valores que requieren revisión, sin sustituirlos ni eliminar registros.

In [ ]:
cy = datos["CONSTRUCTIONYEAR"].astype("float64")
cc = datos["CADCONSTRUCTIONYEAR"].astype("float64")

print("CONSTRUCTIONYEAR con valor:", int(cy.notna().sum()))
print("  menor que 1800:", int((cy < 1800).sum()), "| mayor que 2018:", int((cy > 2018).sum()))
print("CADCONSTRUCTIONYEAR menor que 1800:", int((cc < 1800).sum()))
print()
print("Valores extremos de CONSTRUCTIONYEAR (hasta 30):")
print(cy[(cy < 1800) | (cy > 2018)].value_counts().sort_index().head(30).to_string())
print()
ambos = cy.notna()
dif = cy[ambos] - cc[ambos]
print("Filas con ambos años:", int(ambos.sum()))
print("Años exactamente iguales:", int((dif == 0).sum()))
print("CONSTRUCTIONYEAR - CADCONSTRUCTIONYEAR:")
print(dif.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(1).to_string())

CONSTRUCTIONYEAR con valor: 38942
  menor que 1800: 41 | mayor que 2018: 129
CADCONSTRUCTIONYEAR menor que 1800: 13

Valores extremos de CONSTRUCTIONYEAR (hasta 30):
CONSTRUCTIONYEAR
1.0         1
7.0         1
10.0        2
19.0        3
48.0        1
49.0        2
50.0        2
54.0        1
160.0       1
173.0       1
190.0       1
195.0       1
198.0       4
225.0       1
350.0       1
1056.0      1
1067.0      1
1070.0      1
1073.0      1
1197.0      1
1623.0      2
1627.0      3
1655.0      1
1692.0      1
1696.0      1
1700.0      1
1723.0      1
1730.0      2
1780.0      1
2019.0    114

Filas con ambos años: 38942
Años exactamente iguales: 38464
CONSTRUCTIONYEAR - CADCONSTRUCTIONYEAR:
count    38942.0
mean        -1.1
std         46.6
min      -2001.0
5%           0.0
25%          0.0
50%          0.0
75%          0.0
95%          0.0
max        327.0


### 4.8. Relación entre precio, superficie y precio por metro cuadrado

Comprobamos si el precio por metro cuadrado (`UNITPRICE`) coincide con el precio anunciado (`PRICE`) dividido entre la superficie construida (`CONSTRUCTEDAREA`).

La comparación utiliza una tolerancia numérica para admitir pequeñas diferencias de precisión. En los registros analizados, ambas cantidades coinciden dentro de esa tolerancia.

Esta relación es importante para el modelo: si queremos predecir `UNITPRICE`, no debemos utilizar `PRICE` como variable de entrada, porque junto con la superficie permitiría calcular directamente el objetivo. Esto produciría una fuga de información y una evaluación engañosa.

La superficie sí puede utilizarse como característica del inmueble, siempre que esté disponible en el momento de realizar la predicción.

In [ ]:
u = datos["UNITPRICE"]
for t in [9000, 9500, 9900, 9990]:
    print(f"UNITPRICE mayor que {t}:", int((u > t).sum()))
print()
print("Los 10 valores más altos:")
print(u.sort_values(ascending=False).head(10).round(2).to_string())
print()
calc = datos["PRICE"].astype("float64") / datos["CONSTRUCTEDAREA"].astype("float64")
dif_u = (calc - u).abs()
print("Máxima diferencia |PRICE/CONSTRUCTEDAREA - UNITPRICE|:", float(dif_u.max()))
print("Filas con diferencia mayor que 0.01:", int((dif_u > 0.01).sum()))

UNITPRICE mayor que 9000: 545
UNITPRICE mayor que 9500: 214
UNITPRICE mayor que 9900: 46
UNITPRICE mayor que 9990: 6

Los 10 valores más altos:
79382    9997.56
79287    9994.29
46887    9993.38
2600     9992.25
36579    9991.15
56953    9990.91
2590     9984.92
92912    9979.31
35298    9975.00
11526    9975.00

Máxima diferencia |PRICE/CONSTRUCTEDAREA - UNITPRICE|: 6.366462912410498e-12
Filas con diferencia mayor que 0.01: 0


### 4.9. Coherencia de las categorías de estado del inmueble

Revisamos las columnas que representan el estado del inmueble mediante indicadores binarios.

Comprobamos cuántas categorías están activadas en cada registro para detectar combinaciones que necesiten revisión, como varias categorías incompatibles marcadas a la vez o ninguna categoría seleccionada.

En los datos analizados, cada registro tiene una única categoría activada. Esto muestra coherencia en la codificación, aunque no garantiza que el estado anunciado corresponda al estado real del inmueble.

In [ ]:
bt = datos[["BUILTTYPEID_1", "BUILTTYPEID_2", "BUILTTYPEID_3"]].astype("int64")
print("Cuántos BUILTTYPEID_* valen 1 en cada fila:")
print(bt.sum(axis=1).value_counts().sort_index().to_string())

Cuántos BUILTTYPEID_* valen 1 en cada fila:
1    94815


### Conclusión sobre rangos y coherencia

La revisión permite identificar información redundante y algunos valores que requieren un análisis más detallado.

Las dos variables de aparcamiento coinciden, y el precio por metro cuadrado se obtiene del precio anunciado y la superficie. Estas relaciones se tendrán en cuenta al seleccionar las variables del modelo.

También se han localizado valores geográficos y cantidades de dormitorios que requieren revisión. No se eliminan registros únicamente por presentar valores extremos.

El precio por metro cuadrado alcanza un máximo observado de 9.997,56 €/m². Este resultado describe el archivo analizado, pero no demuestra por sí solo que exista un límite aplicado por la fuente.

## 5. Comparaciones descriptivas iniciales

Exploramos cómo varía el precio anunciado por metro cuadrado entre periodos y entre algunas características de los inmuebles.

Estas comparaciones describen asociaciones en los datos. No permiten afirmar que una característica sea la causa de una diferencia de precio.

### 5.1. Comparación de precios por trimestre

Comparamos el precio por metro cuadrado entre los cuatro periodos disponibles para observar posibles diferencias.

Los trimestres contienen cantidades y composiciones diferentes de anuncios. Además, ningún `ASSETID` aparece en más de un periodo, por lo que no estamos siguiendo el precio de las mismas viviendas.

Por tanto, una diferencia entre trimestres puede reflejar cambios en los inmuebles anunciados, en sus características o en sus precios. No debe interpretarse directamente como una subida o bajada general del mercado.

In [ ]:
por_periodo = datos.groupby("PERIOD").agg(
    filas=("UNITPRICE", "size"),
    mediana=("UNITPRICE", "median"),
    media=("UNITPRICE", "mean"),
    p95=("UNITPRICE", lambda s: s.quantile(0.95)),
)
print(por_periodo.round(0).to_string())

        filas  mediana   media     p95
PERIOD                                
201803  21920   3314.0  3520.0  6639.0
201806  12652   3376.0  3569.0  6542.0
201809  15973   3463.0  3647.0  6769.0
201812  44270   3585.0  3762.0  6877.0


### 5.2. Plantas y disponibilidad de ascensor

Revisamos cómo se distribuyen los anuncios según la planta y la presencia de ascensor. Esta comprobación permite conocer cuántos registros hay disponibles antes de comparar sus precios.

Para interpretar `FLOORCLEAN`, distinguimos las plantas ordinarias de los códigos especiales cuyo significado no esté suficientemente documentado. No debemos asignarles una planta concreta por su valor numérico.

También prestamos atención a los grupos con pocos anuncios. Si una combinación de planta y ascensor tiene escasa representación, sus resultados pueden ser menos estables y deben interpretarse con cautela.

In [ ]:
g = datos.groupby("FLOORCLEAN", observed=True, dropna=False).agg(
    filas=("HASLIFT", "size"),
    pct_ascensor=("HASLIFT", "mean"),
    unitprice_mediana=("UNITPRICE", "median"),
)
g["pct_ascensor"] = (g["pct_ascensor"] * 100).round(1)
g["unitprice_mediana"] = g["unitprice_mediana"].round(0)
print(g.to_string())

            filas  pct_ascensor  unitprice_mediana
FLOORCLEAN                                        
-1            936          60.3             3742.0
0           10112          52.7             2857.0
1           20336          67.8             3351.0
2           17044          64.5             3471.0
3           14754          63.7             3428.0
4           11635          69.9             3622.0
5            6162          88.4             4057.0
6            3733          99.2             4227.0
7            2447          99.9             4000.0
8            1495          99.9             3955.0
9             814         100.0             3502.0
10            450         100.0             3585.0
11           1051          99.5             3831.0
<NA>         3846          61.3             3155.0


### 5.3. Precio por planta y ascensor

Comparamos el precio anunciado por metro cuadrado entre inmuebles con y sin ascensor, agrupados por planta.

Revisamos también la cantidad de registros y de inmuebles distintos de cada grupo. Esto ayuda a valorar si las comparaciones cuentan con suficiente información y evita confundir varios anuncios del mismo inmueble con observaciones independientes.

Las diferencias observadas no representan el efecto aislado del ascensor. Los grupos también pueden diferir en ubicación, superficie, estado y otras características que influyen en el precio.

Esta comparación sirve para formular una hipótesis inicial. En el análisis posterior se contrastará dentro de grupos más comparables.

In [ ]:
d = datos[datos["FLOORCLEAN"].notna()].copy()
d["tramo_planta"] = pd.cut(
    d["FLOORCLEAN"].astype("float64"),
    bins=[-2, 0, 2, 4, 6, 20],
    labels=["-1 a 0", "1 a 2", "3 a 4", "5 a 6", "7 o más"],
)

t = d.groupby(["tramo_planta", "HASLIFT"], observed=True).agg(
    filas=("UNITPRICE", "size"),
    mediana_unitprice=("UNITPRICE", "median"),
)
t["mediana_unitprice"] = t["mediana_unitprice"].round(0)
print(t.to_string())

                      filas  mediana_unitprice
tramo_planta HASLIFT                          
-1 a 0       0         5155             2380.0
             1         5893             3401.0
1 a 2        0        12609             2262.0
             1        24771             3843.0
3 a 4        0         8860             2161.0
             1        17529             4025.0
5 a 6        0          742             2088.0
             1         9153             4215.0
7 o más      0            9             3050.0
             1         6248             3858.0


### 5.4. Recuento inicial de casos para revisión

Aplicamos un primer conjunto de condiciones para localizar registros que merecen una revisión más detallada.

Estas marcas sirven para organizar la exploración. No demuestran que los datos sean incorrectos ni implican que los anuncios deban eliminarse.

Un mismo registro puede cumplir varias condiciones. Por eso, la suma de los recuentos individuales puede superar el número total de filas señaladas.

En los siguientes apartados revisaremos estas condiciones para distinguir situaciones poco habituales de posibles incoherencias.

In [ ]:
# 1) El grupo sin ascensor en plantas 5 a 6: filas frente a inmuebles distintos
sin_asc = d[(d["tramo_planta"] == "5 a 6") & (d["HASLIFT"] == 0)]
print("Plantas 5 a 6 sin ascensor -> filas:", len(sin_asc),
      "| ASSETID distintos:", sin_asc["ASSETID"].nunique())
print()

# 2) Marcas de posibles problemas (solo se cuentan, no se elimina nada)
anio = datos["CONSTRUCTIONYEAR"].astype("float64")
marcas = pd.DataFrame({
    "lejos_de_madrid_mas_15km": datos["DISTANCE_TO_CITY_CENTER"] > 15,
    "habitaciones_11_o_mas": datos["ROOMNUMBER"] >= 11,
    "habitaciones_0_no_estudio": (datos["ROOMNUMBER"] == 0) & (datos["ISSTUDIO"] == 0),
    "banos_0": datos["BATHNUMBER"] == 0,
    "anio_antes_1800": anio < 1800,
    "anio_despues_2018": anio > 2018,
})
print(marcas.sum().to_string())
print()
print("Filas con al menos una marca:", int(marcas.any(axis=1).sum()))
print("Filas con dos o más marcas:", int((marcas.sum(axis=1) >= 2).sum()))

Plantas 5 a 6 sin ascensor -> filas: 742 | ASSETID distintos: 593

lejos_de_madrid_mas_15km       1
habitaciones_11_o_mas         56
habitaciones_0_no_estudio    161
banos_0                       89
anio_antes_1800               41
anio_despues_2018            129

Filas con al menos una marca: 430
Filas con dos o más marcas: 47


### Conclusión de las comparaciones iniciales

La comparación por planta y ascensor muestra diferencias de precio que requieren un análisis más controlado antes de extraer conclusiones.

La cobertura también varía entre grupos. En las plantas 5–6 hay 742 registros sin ascensor, correspondientes a 593 identificadores distintos. En el grupo de plantas 7 o superiores solo hay 9 registros sin ascensor, lo que limita la interpretación de esa comparación.

Las primeras condiciones de revisión señalan 430 registros que cumplen al menos una de ellas. Este recuento es provisional: todavía no representa una decisión de limpieza.

Estos resultados corresponden al dataset completo. No deben confundirse con los análisis posteriores realizados únicamente sobre el conjunto de entrenamiento.

## 6. Revisión de superficie y dormitorios

Analizamos conjuntamente la superficie construida y el número de dormitorios para localizar combinaciones que puedan resultar incoherentes.

Revisar ambas variables aporta más información que observarlas por separado: un número elevado de dormitorios puede ser razonable en una vivienda grande, pero resultar sospechoso en una superficie reducida.

### 6.1. Superficie por dormitorio

Calculamos la superficie construida dividida entre el número de dormitorios para los registros con al menos un dormitorio.

Esta medida no representa el tamaño real de cada dormitorio, porque la superficie construida incluye otras estancias y espacios. Se utiliza únicamente como indicador para localizar casos que necesitan revisión.

Los registros con menos de 5 m² construidos por dormitorio se señalan como especialmente sospechosos. Este umbral es un criterio de exploración, no una norma que demuestre automáticamente que el dato es incorrecto.

Los anuncios con cero dormitorios se revisan por separado, ya que pueden corresponder a estudios y no permiten calcular esta división.

In [ ]:
r = datos[datos["ROOMNUMBER"] > 0].copy()
r["m2_por_hab"] = r["CONSTRUCTEDAREA"].astype("float64") / r["ROOMNUMBER"].astype("float64")

print(r["m2_por_hab"].describe(percentiles=[0.001, 0.01, 0.05, 0.5, 0.95]).round(1).to_string())
print()
for u in [5, 8, 10, 12]:
    print(f"Filas con menos de {u} m² por habitación:", int((r["m2_por_hab"] < u).sum()))
print()
cols = ["ROOMNUMBER", "BATHNUMBER", "CONSTRUCTEDAREA", "PRICE", "ISSTUDIO"]
print(r.sort_values("m2_por_hab").head(10)[cols].to_string())

count    92070.0
mean        39.9
std         19.7
min          1.3
0.1%        14.0
1%          17.7
5%          21.3
50%         35.5
95%         72.0
max        928.0

Filas con menos de 5 m² por habitación: 2
Filas con menos de 8 m² por habitación: 2
Filas con menos de 10 m² por habitación: 9
Filas con menos de 12 m² por habitación: 22

       ROOMNUMBER  BATHNUMBER  CONSTRUCTEDAREA     PRICE  ISSTUDIO
17639          93           2              119  450000.0         0
84060          33           2               90  109000.0         0
1646            3           0               25  119000.0         0
7525            3           1               25   54000.0         0
75515           6           1               55  188000.0         0
49816           3           1               28   59000.0         0
14135           3           1               28   80000.0         0
33411           3           1               28  103000.0         0
14165           3           1               28   69000

### 6.2. Superficies grandes con pocos dormitorios

Revisamos los anuncios con una superficie construida elevada en relación con su número de dormitorios.

Contamos los registros que superan los 100, 150 y 300 m² por dormitorio y mostramos algunos ejemplos de los que superan los 150 m².

Estos umbrales sirven para explorar los casos extremos, no para clasificarlos automáticamente como errores. Una vivienda puede tener pocos dormitorios y una superficie amplia destinada a salones, otras estancias o espacios incluidos en la superficie construida.

Por tanto, examinamos también los baños, el precio y el indicador de estudio antes de decidir si estos registros necesitan una revisión adicional.

In [ ]:
for u in [100, 150, 300]:
    print(f"Filas con más de {u} m² por habitación:", int((r["m2_por_hab"] > u).sum()))
print()

alt = r[r["m2_por_hab"] > 150].sort_values("m2_por_hab", ascending=False)
cols = ["ROOMNUMBER", "BATHNUMBER", "CONSTRUCTEDAREA", "PRICE", "UNITPRICE", "ISSTUDIO"]
print(alt.head(10)[cols].to_string())

Filas con más de 100 m² por habitación: 839
Filas con más de 150 m² por habitación: 130
Filas con más de 300 m² por habitación: 33

       ROOMNUMBER  BATHNUMBER  CONSTRUCTEDAREA      PRICE    UNITPRICE  ISSTUDIO
42805           1           1              928  2732000.0  2943.965517         0
63226           1           1              926  4098000.0  4425.485961         0
63544           1           1              922  3060000.0  3318.872017         0
74176           1           1              800   747000.0   933.750000         0
34925           1           1              732  4925000.0  6728.142077         0
75578           1           1              724  2307000.0  3186.464088         0
81665           1           1              712  1888000.0  2651.685393         0
78518           1           1              650   652000.0  1003.076923         0
75330           1           1              613  2065000.0  3368.678630         0
23378           1           1              600   867000.0 

### 6.3. Superficie según número de dormitorios

Comparamos la distribución de la superficie construida entre los anuncios de uno a seis dormitorios. Revisamos la mediana, los percentiles altos y los valores máximos para conocer las diferencias entre grupos.

También contamos los registros con más de 300 m² según su número de dormitorios. En este último recuento, el valor 8 agrupa los anuncios con ocho o más dormitorios.

Esta revisión ayuda a identificar combinaciones poco habituales, como superficies muy grandes con pocos dormitorios. Sin embargo, no permite determinar por sí sola si la superficie o el número de dormitorios son incorrectos.

Estas combinaciones se utilizarán como señales de revisión, sin eliminar automáticamente los registros.

In [ ]:
g = datos[datos["ROOMNUMBER"].between(1, 6)].groupby("ROOMNUMBER")["CONSTRUCTEDAREA"]
print("Superficie (m²) según número de habitaciones:")
print(g.describe(percentiles=[0.5, 0.95, 0.99]).round(0).to_string())
print()

grandes = datos[datos["CONSTRUCTEDAREA"] > 300]
print("Filas con más de 300 m², según ROOMNUMBER (8 = 8 o más):")
print(grandes["ROOMNUMBER"].clip(upper=8).value_counts().sort_index().to_string())

Superficie (m²) según número de habitaciones:
              count   mean    std   min    50%    95%    99%    max
ROOMNUMBER                                                         
1           13338.0   56.0   32.0  21.0   51.0   91.0  135.0  928.0
2           28422.0   76.0   28.0  21.0   70.0  126.0  173.0  950.0
3           33961.0  102.0   42.0  25.0   93.0  177.0  250.0  985.0
4           11674.0  161.0   67.0  52.0  144.0  299.0  385.0  951.0
5            3350.0  236.0  100.0  78.0  212.0  430.0  572.0  941.0
6             789.0  296.0  140.0  55.0  260.0  540.0  805.0  982.0

Filas con más de 300 m², según ROOMNUMBER (8 = 8 o más):
ROOMNUMBER
0     24
1     31
2      9
3    117
4    507
5    678
6    319
7    122
8    136


## 7. Marcas de revisión y comparación descriptiva

Tras las comprobaciones anteriores, reunimos cuatro condiciones para identificar registros que necesitan atención:

- Información geográfica anómala.
- Superficie construida por dormitorio especialmente reducida.
- Más de 300 m² de superficie con un máximo de dos dormitorios.
- Número de baños igual a cero.

Creamos estas marcas en una copia de los datos para conservar la tabla original. Un registro puede cumplir varias condiciones, por lo que contamos también cuántas filas presentan al menos una marca.

La variable técnica `flag_hab_imposible` identifica una combinación muy poco plausible de superficie y dormitorios. Su nombre no significa que hayamos demostrado que el dato sea imposible.

Finalmente, comparamos descriptivamente los registros señalados con los restantes. Esta comparación permite conocer sus características, pero no demuestra que eliminarlos mejore el modelo ni constituye una limpieza definitiva.

In [ ]:
datos_rev = datos.copy()
area = datos_rev["CONSTRUCTEDAREA"].astype("float64")
hab = datos_rev["ROOMNUMBER"].astype("float64")

datos_rev["flag_fuera_madrid"] = datos_rev["DISTANCE_TO_CITY_CENTER"] > 50
datos_rev["flag_hab_imposible"] = (hab > 0) & ((area / hab) < 5)
datos_rev["flag_sup_grande_pocas_hab"] = (area > 300) & (hab <= 2)
datos_rev["flag_banos_0"] = datos_rev["BATHNUMBER"] == 0

flags = [c for c in datos_rev.columns if c.startswith("flag_")]
print(datos_rev[flags].sum().to_string())

datos_rev["alguna_flag"] = datos_rev[flags].any(axis=1)
print()
print("Filas con alguna marca:", int(datos_rev["alguna_flag"].sum()))
print()
print("UNITPRICE según tenga o no alguna marca:")
print(datos_rev.groupby("alguna_flag")["UNITPRICE"].agg(["size", "mean", "median"]).round(1).to_string())

flag_fuera_madrid             1
flag_hab_imposible            2
flag_sup_grande_pocas_hab    64
flag_banos_0                 89

Filas con alguna marca: 152

UNITPRICE según tenga o no alguna marca:
              size    mean  median
alguna_flag                       
False        94663  3661.6  3480.0
True           152  3323.1  3170.2


### Conclusión de las marcas de revisión

Las cuatro condiciones señalan 152 registros distintos que cumplen al menos una de ellas:

- 1 registro con información geográfica anómala.
- 2 registros con una superficie por dormitorio especialmente reducida.
- 64 registros con más de 300 m² y un máximo de dos dormitorios.
- 89 registros con cero baños.

La suma de estas cantidades supera los 152 registros porque algunas filas cumplen varias condiciones.

Este recuento sustituye al conjunto inicial de condiciones de revisión, que señalaba 430 registros. La diferencia responde a los criterios utilizados, no a una eliminación de filas.

En esta exploración conservamos todos los registros. El tratamiento definitivo de los casos señalados se justificará en el notebook de modelado.

## 8. Exportación tabular y comprobación

Guardamos la copia tabular en formato CSV para utilizarla en las siguientes fases del proyecto.

El archivo conserva los 94.815 registros y las 41 columnas originales restantes tras retirar la geometría redundante. No incluye las marcas de revisión creadas en la copia auxiliar ni aplica la limpieza definitiva del modelado.

Después de guardar el archivo, lo volvemos a cargar y comprobamos sus dimensiones para verificar que la exportación conserva el número de filas y columnas esperado.

El formato CSV no conserva necesariamente los tipos de datos de pandas, como las categorías. Por ello, al reutilizarlo será necesario revisar los tipos de las columnas.

La ruta de guardado es relativa al directorio desde el que se ejecuta el notebook. Si se ejecuta en Colab, el archivo se genera en su entorno de ejecución y debe descargarse o guardarse para conservarlo.

In [ ]:
ruta_csv = Path("project/data/Madrid_Sale_original.csv")
ruta_csv.parent.mkdir(parents=True, exist_ok=True)

datos.to_csv(ruta_csv, index=False, encoding="utf-8")

comprobacion = pd.read_csv(ruta_csv)
assert comprobacion.shape == datos.shape
assert comprobacion.columns.tolist() == datos.columns.tolist()

print("CSV guardado y comprobado:", comprobacion.shape)
print("Ruta:", ruta_csv)

CSV guardado y comprobado: (94815, 41)
Ruta: project/data/Madrid_Sale_original.csv


In [ ]:
from google.colab import files
files.download(str(ruta_csv))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Conclusión y continuidad

La exploración ha permitido conocer la estructura, la calidad y las principales limitaciones del dataset de anuncios de venta de Madrid de 2018.

La tabla exportada conserva 94.815 registros y 41 columnas. No se han eliminado anuncios durante esta exploración; únicamente se ha retirado la geometría redundante.

Las principales conclusiones son:

- Existen valores ausentes, especialmente en el año de construcción declarado.
- No hay duplicados exactos, pero algunos identificadores aparecen varias veces dentro del mismo trimestre.
- Se han señalado 152 registros para revisión mediante cuatro condiciones. Estas marcas no implican que todos sean incorrectos.
- Algunas variables contienen información redundante.
- El precio anunciado, junto con la superficie, permite calcular el precio por metro cuadrado. Por ello, `PRICE` debe excluirse como predictor si el objetivo es `UNITPRICE`.

Las comparaciones descriptivas sirven para formular hipótesis, pero no demuestran relaciones causales. Además, los datos representan precios anunciados de 2018, no precios de compraventa cerrada ni el mercado actual.

### Continuidad del proyecto

El notebook de modelado documentará la limpieza definitiva, la división temporal de los datos, los métodos de referencia y la evaluación de los modelos.

Esta exploración inicial ha utilizado el dataset completo. Por transparencia, se reconoce que sus características generales ya se han observado; los ajustes del modelo deberán realizarse con entrenamiento y validación, reservando la prueba final para la evaluación.

## 10. Fuentes

- **paezha (s. f.).** *idealista18*. Repositorio del dataset y documentación general. Se utiliza la versión identificada por el commit `e9ef80618f8a9cc28f672ac138f298bbd519918e`.
  https://github.com/paezha/idealista18/tree/e9ef80618f8a9cc28f672ac138f298bbd519918e

- **paezha (s. f.).** *Madrid_Sale: diccionario de datos*. Documentación utilizada para interpretar las variables.
  https://github.com/paezha/idealista18/blob/e9ef80618f8a9cc28f672ac138f298bbd519918e/man/Madrid_Sale.Rd

- **Open Data Commons (s. f.).** *Open Database License (ODbL) v1.0*. Condiciones de la licencia indicada por el repositorio de origen.
  https://opendatacommons.org/licenses/odbl/1-0/

### Documento relacionado

El planteamiento del problema, los objetivos y el alcance se recogen en el [primer Project Brief](https://github.com/D-F-V/Project_brief/blob/main/project/project-brief.md).

### Nota sobre la documentación

El diccionario presenta cifras de dimensiones que no coinciden con el archivo utilizado. En este notebook se toman como referencia las dimensiones comprobadas al cargar los datos: 94.815 registros y 42 columnas originales, que pasan a 41 al retirar la geometría redundante.

Las descripciones de la fuente ayudan a interpretar las variables, pero no resuelven todas las dudas sobre los códigos y los valores detectados.